# Functional Enrichment

Analysis of peaks for gene set enrichment with rGREAT.

### Build Gene Database

In [ ]:
# Initialize
library(rtracklayer)
library(GenomicRanges)
library(rGREAT)

# Move to working directory
setwd("/home/dalbao/AlbaoRunx3Manuscript/cutnrun")

gtf <- import("source_data/genes.gtf")
tx  <- gtf[gtf$type == "transcript"]

# --- Filter to genes that can plausibly appear in your gene sets ---
keep_bt <- c("protein_coding", "lncRNA")     # add "TR_*"/"IG_*" if relevant to you
tx <- tx[tx$gene_biotype %in% keep_bt]

# Optional: drop poorly-supported isoforms so a spurious transcript
# doesn't drag the TSS upstream. TSL 1-3 is a common cut.
# tx <- tx[tx$transcript_support_level %in% c("1","2","3","NA")]

tx <- tx[!is.na(tx$gene_name) & tx$gene_name != ""]

# --- Collapse to one most-5' TSS per gene (strand-aware) ---
spans <- range(split(tx, tx$gene_name))
spans <- spans[elementNROWS(spans) == 1]     # drop multi-strand/scaffold artifacts
gene_span <- unlist(spans)

tss <- resize(gene_span, width = 1, fix = "start")
mcols(tss) <- NULL
tss$gene_id <- names(gene_span)              # SYMBOL, to match gene sets
names(tss) <- NULL
tss <- tss[!is.na(tss$gene_id) & tss$gene_id != ""]
tss <- tss[!duplicated(tss$gene_id)]

length(tss)     # expect ~20-35k with the filter above, not ~55k

[1] 21913

In [ ]:
head(tss)

GRanges object with 6 ranges and 1 metadata column:
      seqnames    ranges strand |       gene_id
         <Rle> <IRanges>  <Rle> |   <character>
  [1]       12  85824550      - | 0610007P14Rik
  [2]       11  51688874      - | 0610009B22Rik
  [3]       11 120348678      + | 0610009L18Rik
  [4]       18  38250249      + | 0610009O20Rik
  [5]       11  23633639      - | 0610010F05Rik
  [6]       11  70237914      - | 0610010K14Rik
  -------
  seqinfo: 73 sequences from an unspecified genome; no seqlengths

### Peak Import and Chromosome End Attachment

In [ ]:
# Load Peaks
peaks.all <- import("01_peakEDA/all.clean.noCluster.bed")
peaks.cl1 <- import("01_peakEDA/cluster1.clean.noCluster.bed")
peaks.cl2 <- import("01_peakEDA/cluster2.clean.noCluster.bed")

# Restrict to primary chromosomes; scaffolds add noise and no power
main <- c(1:19, "X")   # mm10/mm39
tss   <- keepSeqlevels(tss, main, pruning.mode = "coarse")
peaks.all <- keepSeqlevels(peaks.all, main, pruning.mode = "coarse")
peaks.cl1 <- keepSeqlevels(peaks.cl1, main, pruning.mode = "coarse")
peaks.cl2 <- keepSeqlevels(peaks.cl2, main, pruning.mode = "coarse")

# Attach chromosome lengths to TSS GRanges object, so GREAT can work
fai <- read.table("source_data/260713_reBAM2_noDeDup_CPM/04_reporting/igv/genome.fa.fai")[, 1:2]
sl  <- setNames(fai$V2, fai$V1)
seqlengths(tss) <- sl[seqlevels(tss)]

### Process Gene Sets

In [ ]:
# Load gene sets
gene.sets <- read.csv("/home/dalbao/CommonGeneSets/ConvertedLists/v11/genesets_v11_Ensembl98.csv", header = TRUE, stringsAsFactors = FALSE)
head(gene.sets)

,gs_name,gene_symbol,EnsemblID
,<chr>,<chr>,<chr>
1,TXM_lib-8C,Abi2,ENSMUSG00000026782
2,TXM_lib-8C,Acsf3,ENSMUSG00000015016
3,TXM_lib-8C,Adam8,ENSMUSG00000025473
4,TXM_lib-8C,Ano10,ENSMUSG00000037949
5,TXM_lib-8C,Cables1,ENSMUSG00000040957
6,TXM_lib-8C,Cd9,ENSMUSG00000030342


In [ ]:
# Keep only gene_symbol if it exists in the TSS object
gene.sets <- gene.sets[gene.sets$gene_symbol %in% tss$gene_id, ]

# Process gene sets into a list of unique gene symbols per gene set name
gene.sets <- split(gene.sets$gene_symbol, gene.sets$gs_name)
gene.sets <- lapply(gene.sets, unique)

In [ ]:
tss$gene_id[tss$gene_id %in% c("Sept6", "Septin6")]  # check that Runx3 is present in the TSS object
gene.sets$gs_name[grep("Sept", gene.sets$gs_name)]  # check that Runx3 is present in the gene sets``

[1] "Sept6"

NULL

In [ ]:
res.all <- great(
    gr                = peaks.all,
    gene_sets         = gene.sets,
    tss_source        = tss,
    mode              = "basalPlusExt",   # GREAT default
    basal_upstream    = 5000,
    basal_downstream  = 1000,
    extension         = 1000000,
    min_gene_set_size = 5,
    exclude           = NULL,             # "gap" needs a recognized genome
    cores             = 4
)

res.cl1 <- great(
    gr                = peaks.cl1,
    gene_sets         = gene.sets,
    tss_source        = tss,
    mode              = "basalPlusExt",   # GREAT default
    basal_upstream    = 5000,
    basal_downstream  = 1000,
    extension         = 1000000,
    min_gene_set_size = 5,
    exclude           = NULL,             # "gap" needs a recognized genome
    cores             = 4
)

res.cl2 <- great(
    gr                = peaks.cl2,
    gene_sets         = gene.sets,
    tss_source        = tss,
    mode              = "basalPlusExt",   # GREAT default
    basal_upstream    = 5000,
    basal_downstream  = 1000,
    extension         = 1000000,
    min_gene_set_size = 5,
    exclude           = NULL,             # "gap" needs a recognized genome
    cores             = 4
)

* TSS extension mode is 'basalPlusExt'.

* construct the basal domains by extending 5000bp to upstream and 1000bp to downsteram of TSS.

* calculate distances to neighbour regions.



* extend to both sides until reaching the neighbour genes or to the maximal extension.

* check gene ID type in `gene_sets` and in `extended_tss`.

* use whole genome as background.

* overlap `gr` to background regions (based on midpoint).

* in total 4172 `gr`.

* overlap extended TSS to background regions.

* check which genes are in the gene sets.

* only take gene sets with size >= 5.

* in total 186 gene sets.

* overlap `gr` to every extended TSS.

* perform binomial test for each biological term.

* TSS extension mode is 'basalPlusExt'.

* construct the basal domains by extending 5000bp to upstream and 1000bp to downsteram of TSS.

* calculate distances to neighbour regions.

* extend to both sides until reaching the neighbour genes or to the maximal extension.

* check gene ID type in `gene_sets` and in `extended_tss`.

* use whole genome as background.

* overlap `gr` to background regions (based on midpoint).

* in total 1190 `gr`.

* overlap extended TSS to background regio

In [ ]:
tb.all <- getEnrichmentTable(res.all)
tb.cl1 <- getEnrichmentTable(res.cl1)
tb.cl2 <- getEnrichmentTable(res.cl2)

In [ ]:
# Make folder if it does not exist
if (!dir.exists("05_enrichment")) {
    dir.create("05_enrichment", recursive = TRUE)
}

saveData <- function(df, filename){
    write.csv(df, paste0("05_enrichment/", filename))
}

saveData(tb.all , "all.peaks.enrichment.csv")
saveData(tb.cl1 , "cl1.peaks.enrichment.csv")
saveData(tb.cl2 , "cl2.peaks.enrichment.csv")

In [ ]:
assoc_all <- getRegionGeneAssociations(res.all)
assoc_cl1 <- getRegionGeneAssociations(res.cl1)
assoc_cl2 <- getRegionGeneAssociations(res.cl2)

In [ ]:
head(assoc_all)

GRanges object with 6 ranges and 3 metadata columns:
      seqnames            ranges strand |             name annotated_genes
         <Rle>         <IRanges>  <Rle> |      <character> <CharacterList>
  [1]        1   7397451-7398450      * |  shRunx3_Runx1_5          Pcmtd1
  [2]        1   9847901-9848700      * |  shCd19_Runx1_27     Sgk3,Mcmdc2
  [3]        1 10037651-10038650      * |   shCd19_Runx3_1     Cspp1,Cops5
  [4]        1 13371951-13373200      * |   shCd19_Runx3_2    Prdm14,Ncoa2
  [5]        1 13374001-13374750      * |  shCd19_Runx1_46           Ncoa2
  [6]        1 13382101-13382950      * | shRunx3_Runx3_33     Ncoa2,Tram1
        dist_to_TSS
      <IntegerList>
  [1]        308531
  [2]  49794,-59938
  [3]           0,0
  [4]   -244788,883
  [5]             0
  [6]  -8018,206960
  -------
  seqinfo: 20 sequences from an unspecified genome; no seqlengths

### Region Hit BED Files

For each significantly enriched gene set (p_adjust < 0.05, binomial/region-based test)
in each cluster, export a BED file of the peaks ("observed region hits") whose
GREAT regulatory domain is annotated to a gene in that set.

In [ ]:
# Significant gene sets per cluster (binomial/region-based test, matches observed_region_hits)
sig_sets <- function(tb) tb$id[tb$p_adjust < 0.05]

sets.all <- sig_sets(tb.all)
sets.cl1 <- sig_sets(tb.cl1)
sets.cl2 <- sig_sets(tb.cl2)

c(all = length(sets.all), cl1 = length(sets.cl1), cl2 = length(sets.cl2))

all cl1 cl2 
148 110 134

In [ ]:
# Make a filesystem-safe filename from a gene set name
sanitize <- function(x) gsub("[^A-Za-z0-9._-]+", "_", x)

# Export one BED per gene set: the peaks whose annotated genes overlap that set
exportRegionHits <- function(assoc, sig_ids, out_dir) {
    if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)
    for (id in sig_ids) {
        genes <- gene.sets[[id]]
        is_hit <- vapply(assoc$annotated_genes, function(g) any(g %in% genes), logical(1))
        hit <- assoc[is_hit]
        if (length(hit) == 0) next
        export.bed(hit, paste0(out_dir, "/", sanitize(id), ".bed"))
    }
}

exportRegionHits(assoc_all, sets.all, "05_enrichment/regionHits/all")
exportRegionHits(assoc_cl1, sets.cl1, "05_enrichment/regionHits/cl1")
exportRegionHits(assoc_cl2, sets.cl2, "05_enrichment/regionHits/cl2")

### Gene Hits for a Single Gene Set

For `Albao_Runx3OE` specifically: which genes in the set have at least one observed
region hit (a peak whose GREAT regulatory domain is annotated to that gene), and
which peak(s) support each gene, in each of `all`/`cl1`/`cl2`.

In [ ]:
geneHits <- function(assoc, set_name) {
    genes <- gene.sets[[set_name]]
    df <- data.frame(
        peak = rep(assoc$name, elementNROWS(assoc$annotated_genes)),
        gene = unlist(assoc$annotated_genes)
    )
    df <- df[df$gene %in% genes, ]
    agg <- aggregate(peak ~ gene, df, function(x) paste(sort(unique(x)), collapse = ","))
    agg$n_peaks <- lengths(strsplit(agg$peak, ","))
    agg[order(-agg$n_peaks, agg$gene), c("gene", "n_peaks", "peak")]
}
hits_geneset <- function(gs_name){
    hits.all <- geneHits(assoc_all, gs_name)
    hits.cl1 <- geneHits(assoc_cl1, gs_name)
    hits.cl2 <- geneHits(assoc_cl2, gs_name)

    if (!dir.exists("05_enrichment/geneHits")) dir.create("05_enrichment/geneHits", recursive = TRUE)
    write.csv(hits.all, paste0("05_enrichment/geneHits/", gs_name, ".all.csv"), row.names = FALSE)
    write.csv(hits.cl1, paste0("05_enrichment/geneHits/", gs_name, ".cl1.csv"), row.names = FALSE)
    write.csv(hits.cl2, paste0("05_enrichment/geneHits/", gs_name, ".cl2.csv"), row.names = FALSE)

    cat(sprintf(gs_name, "gene set size: %d\n", length(gene.sets[[gs_name]])))
    c(all = nrow(hits.all), cl1 = nrow(hits.cl1), cl2 = nrow(hits.cl2))
}

hits_geneset("2016_SCIENCE_Mackay---TEM_vs_TCM-TRM")  # regulation of

Warning message in sprintf(gs_name, "gene set size: %d\n", length(gene.sets[[gs_name]])):
“2 arguments not used by format '2016_SCIENCE_Mackay---TEM_vs_TCM-TRM'”


2016_SCIENCE_Mackay---TEM_vs_TCM-TRM

all cl1 cl2 
 73  30  58

### Bubble Plot: Selected Gene Sets

Fold enrichment (fill) and adjusted p-value (size, as -log10) for `all`/`cl1`/`cl2`,
restricted to a curated set of gene sets and to results that are `Significant`
(p_adjust < 0.05).

In [ ]:
library(ggplot2)
library(viridis)
source("../scripts/plotSEA.R")

tb.all$group <- "all peaks"
tb.cl1$group <- "type 1 peaks"
tb.cl2$group <- "type 2 peaks"

bubble_df <- rbind(tb.all, tb.cl1, tb.cl2)
bubble_df$group <- factor(bubble_df$group, levels = c("all peaks", "type 1 peaks", "type 2 peaks"))

# Gene sets to display: real name -> pretty name, in plotting order (top to bottom)
gene_set_labels <- c(
    "Bresser_hdTcm"                                = "Bresser hdTcm",
    "2016_SCIENCE_Mackay---TCM_vs_TEM-TRM"          = "Mackay Tcm",
    "2016_SCIENCE_Mackay---TRM_vs_TCM-TEM"          = "Mackay Trm",
    "2016_SCIENCE_Mackay---TEM_vs_TCM-TRM"          = "Mackay Tem",
    "Albao_Runx3OE"                                 = "RUNX3 OE up",
    "Albao_Runx3KD_Down"                            = "shRunx3 Down",
    "Albao_Runx3OE_Down"                            = "RUNX3 OE Down",
    "Albao_Runx3KD"                                 = "shRunx3 Up"
)

# Keep only the curated gene sets, and only results that are Significant
bubble_df <- bubble_df[bubble_df$id %in% names(gene_set_labels) & bubble_df$p_adjust < 0.05, ]
bubble_df$id <- factor(gene_set_labels[bubble_df$id], levels = rev(gene_set_labels))

pdf(file = "05_enrichment/GREAT.pdf", width = 4, height = 4)
plotGREATbubble(bubble_df, title = "GREAT Gene Set Enrichment")
dev.off()

pdf 
  2